Evidencia Integradora: Red Neuronal para detectar Señales de Emergencia

Gabriel Pérez Bárcenas 9° "B"

In [ ]:
# Etapa 1: Declarar las clases necesarias
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Activation, Conv2D, MaxPooling2D
from PIL import Image
import os

def check_images(directory):
    for root, dirs, files in os.walk(directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            try:
                img = Image.open(file_path)
                img.verify()  # Verifica que la imagen sea válida
            except (IOError, SyntaxError) as e:
                print(f'Archivo corrupto: {file_path}')

# Ruta al directorio de entrenamiento
folder_path = "/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Entrenamiento"
check_images(folder_path)

# Ruta al directorio de validación
folder_path_validacion = "/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Validacion"
check_images(folder_path_validacion)

#Etapa 2: Declarar los datos de entrenamiento y validacion
entrenamiento="/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Entrenamiento"
validacion="/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Validacion"

epocas = 70
altura,anchura = 180,180
batch = 4
pasos = 100
kernel1=32
kernel2=64
kernel1_size=(3,3)
kernel2_size=(4,4)
pooling=(2,2)
clases=7

#Etapa 3: Datos sinteticos (opcional, si tengo pocos datos)

sinteticos_train = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

sinteticos_valid = ImageDataGenerator(rescale=1./255)

# Leer imágenes de la fuente
imagenes_entrenamiento = sinteticos_train.flow_from_directory(
    entrenamiento,
    target_size=(altura, anchura),
    batch_size=batch,
    class_mode='categorical'
)

imagenes_validacion = sinteticos_valid.flow_from_directory(
    validacion,
    target_size=(altura, anchura),
    batch_size=batch,
    class_mode='categorical'
)

Found 217 images belonging to 7 classes.
Found 34 images belonging to 7 classes.


In [30]:
# Etapa 4: Definir el modelo
CNN = Sequential()

# Primera capa Convolucional
CNN.add(Conv2D(kernel1, kernel1_size, padding="same", input_shape=(altura, anchura, 3), activation="relu"))
CNN.add(MaxPooling2D(pool_size=pooling))

# Segunda capa Convolucional
CNN.add(Conv2D(kernel2, kernel2_size, padding="same", activation="relu"))
CNN.add(MaxPooling2D(pool_size=pooling))

# Aplicar flatten
CNN.add(Flatten())

# Conectar la capa de convolucion con la red neuronal multicapa
CNN.add(Dense(512, activation="relu"))

# Apagar un % de neuronas
CNN.add(Dropout(0.5))

# Capa de salida
CNN.add(Dense(clases, activation='softmax'))  # Número de clases


In [31]:
# Etapa 5: Establecer los parámetros iniciales del entrenamiento
CNN.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["acc", "mse"])

In [32]:
# Etapa 6: Entrenar el modelo
historia = CNN.fit(imagenes_entrenamiento,steps_per_epoch=pasos,epochs=epocas,validation_data=imagenes_validacion,validation_steps=pasos)

Epoch 1/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - acc: 0.3344 - loss: 4.3986 - mse: 0.1284 - val_acc: 0.4118 - val_loss: 1.3022 - val_mse: 0.0946
Epoch 2/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.6416 - loss: 0.9900 - mse: 0.0748 - val_acc: 0.5588 - val_loss: 1.7467 - val_mse: 0.0848
Epoch 3/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - acc: 0.7373 - loss: 0.6805 - mse: 0.0537 - val_acc: 0.6765 - val_loss: 1.2330 - val_mse: 0.0656
Epoch 4/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - acc: 0.7677 - loss: 0.7137 - mse: 0.0525 - val_acc: 0.6471 - val_loss: 2.7595 - val_mse: 0.0887
Epoch 5/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - acc: 0.7321 - loss: 0.7965 - mse: 0.0523 - val_acc: 0.7647 - val_loss: 1.1254 - val_mse: 0.0521
Epoch 6/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - acc: 0.8451 - loss: 0.3784 - mse: 0.0288 - val_acc: 0.7647 - val_loss: 1.0561 - val_mse: 0.0572
Epoch 7/70
100/100 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - acc: 0.8543 - loss: 0.3416 - mse: 0.028

In [ ]:
#Etapa 7: Guardar el Modelo

# Guardar la arquitectura
CNN.save("/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Modelo/CNN.keras")

# Guardar los pesos
CNN.save_weights("/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Modelo/Pesos.weights.h5")


In [ ]:
#Etapa 8: Evaluar modelo entrenado en clase

import numpy as np
from keras.utils import load_img, img_to_array
from keras.models import load_model

clases_nombres = ['Alarma de Emergencia', 'Alto Voltaje', 'Cubrebocas', 'No fumar', 'Punto de Reunion', 'Salida de Emergencia','Sustancia Toxica']

# Ruta de la imagen y del modelo
evaluar = "/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Validacion/No Fumar/33.jpg"
modelo = "/content/drive/MyDrive/Red_Neuronal_de_Señales_de_Emergencia/Modelo/CNN.keras"

# Parámetros de la imagen
altura, anchura = 180,180
# Cargar el modelo (esto incluye la arquitectura y los pesos)
CNN = load_model(modelo)

# Leer y preparar la imagen para la predicción
imagen = load_img(evaluar, target_size=(altura, anchura))
imagen = img_to_array(imagen)
imagen = np.expand_dims(imagen, axis=0)

# Evaluar la imagen con el modelo
prediccion = CNN.predict(imagen)
indice_clase = np.argmax(prediccion)
clase = clases_nombres[indice_clase]
print(f"Predicción: {clase} con probabilidad {prediccion[0][indice_clase]:.2f}")